# Setup

In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def parse_value(value_str):
    if isinstance(value_str, str):
        value_str = value_str.replace('€', '')
        if 'M' in value_str:
            return float(value_str.replace('M', '')) * 1000000
        elif 'K' in value_str:
            return float(value_str.replace('K', '')) * 1000
    return float(value_str)

def standardize_data(X):
    means = np.nanmean(X, axis=0)
    stds = np.nanstd(X, axis=0)
    stds[stds == 0] = 1e-8 # Prevent division by zero for constants
    return (X - means) / stds

In [ ]:
# Calculates Euclidean distance across all players (lower is more similar)
def euclidean_distance(target, pool):
    return np.sqrt(np.sum((pool - target) ** 2, axis=1))

# Calculates Manhattan distance across all players (lower is more similar).
def manhattan_distance(target, pool):
    return np.sum(np.abs(pool - target), axis=1)

# Calculates Cosine similarity across all players (higher is more similar)
def cosine_similarity(target, pool):
    dot_product = np.sum(pool * target, axis=1)
    norm_target = np.linalg.norm(target)
    norm_pool = np.linalg.norm(pool, axis=1)
    return dot_product / (norm_target * norm_pool + 1e-8)

# Returns indices of the top 5 most similar players, excluding the target
def get_top_5_similar(target_idx, scores, metric_type="distance"):
    if metric_type == "distance":
        sorted_indices = np.argsort(scores) # Ascending
    else:
        sorted_indices = np.argsort(scores)[::-1] # Descending
    
    # Remove the target player from their own search results
    sorted_indices = sorted_indices[sorted_indices != target_idx]
    return sorted_indices[:5]

In [ ]:
# Loading the data
df = pd.read_csv('kl.csv', encoding='cp1252') 

# Feature selection
attributes = ['Finishing', 'ShortPassing', 'Dribbling', 'SprintSpeed', 
              'Strength', 'Stamina', 'Interceptions', 'StandingTackle', 'Value']

# Clean the Value column
df['Value'] = df['Value'].apply(parse_value)

# Extract raw data and target
X_raw = df[attributes].values


# Find Salah's index
target_idx = df[df['Name'] == 'M. Salah'].index[0]
target_raw = X_raw[target_idx]

# Standardization Experiment
X_std = standardize_data(X_raw)
target_std = X_std[target_idx]

# Compute Metrics
dist_euclidean = euclidean_distance(target_std, X_std)
dist_manhattan = manhattan_distance(target_std, X_std)
sim_cosine = cosine_similarity(target_std, X_std)

# Extract Top 5
top5_euclidean = get_top_5_similar(target_idx, dist_euclidean, "distance")
top5_manhattan = get_top_5_similar(target_idx, dist_manhattan, "distance")
top5_cosine = get_top_5_similar(target_idx, sim_cosine, "similarity")

# Display Results
print("Euclidean Shortlist:", df.iloc[top5_euclidean]['Name'].values)
print("Manhattan Shortlist:", df.iloc[top5_manhattan]['Name'].values)
print("Cosine Shortlist:", df.iloc[top5_cosine]['Name'].values)

Euclidean Shortlist: ['Coutinho' 'A. Griezmann' 'J. Rodríguez' 'L. Sané' 'Cristiano Ronaldo']
Manhattan Shortlist: ['J. Rodríguez' 'A. Griezmann' 'K. Mbappé' 'Coutinho' 'L. Sané']
Cosine Shortlist: ['O. Marrufo' 'M. Chergui' 'H. Al Mansour' 'Hernáiz' 'E. Guerrero']
